# Proposed Model Testing -- Bad Image to Good Image

Final demo/testing notebook. Loads the trained **proposed model**
(`DistributionMixtureRestorationNet`, the Full model with LDMH + FiLM) and
runs the actual restoration: **Corrupt (y) -> Restored (x-hat)**, matching
the hackathon's `f: x -> y` framing (True -> Corrupt), just inverted.

This notebook:
1. Loads the trained checkpoint.
2. Shows single-image and multi-image before/after demos (with a
   histogram comparison, in the same style as the competition's own
   slides).
3. Runs a quantitative sanity check on the labeled validation split
   (PSNR/SSIM, since ground truth exists there).
4. Batch-processes the ENTIRE real competition test set (`Test_NoisyLR`,
   no ground truth) and saves every restored output as `.npy` --
   this is the submission-ready deliverable.

In [1]:
import sys, os, time
sys.path.insert(0, "..")

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from utils.data import match_pairs, split_pairs, list_npy, load_npy, downsample
from utils.metrics import psnr_np, ssim_np
from models.restoration_net import DistributionMixtureRestorationNet

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda


In [2]:
# ============================================================
# CONFIG
# ============================================================
TRAIN_GT_DIR = r"C:\Users\DELL\OneDrive\Desktop\Golu Kumar\Golu Kumar\Semi conductor image paper\train\train\GT"
TRAIN_NOISY_DIR = r"C:\Users\DELL\OneDrive\Desktop\Golu Kumar\Golu Kumar\Semi conductor image paper\train\train\NoisyLR"
TEST_NOISY_DIR = r"C:\Users\DELL\OneDrive\Desktop\Golu Kumar\Golu Kumar\Semi conductor image paper\Test_NoisyLR\NoisyLR"

CKPT_PATH = "../results/checkpoints/DistributionMixtureRestorationNet.pth"   # the proposed Full model
OUTPUT_DIR = "../results/test_predictions"    # submission-ready restored .npy files
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs("../results/demo_figures", exist_ok=True)


## Load the trained proposed model

In [3]:
model = DistributionMixtureRestorationNet(base_ch=32, n_components=3,
                                            n_lr_blocks=4, n_hr_blocks=2, use_film=True)

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded checkpoint from epoch {ckpt['epoch']}")
    print(f"  best val PSNR: {ckpt['best_val_psnr']:.3f}")
    print(f"  best val SSIM: {ckpt['best_val_ssim']:.4f}")
else:
    raise FileNotFoundError(
        f"Checkpoint not found at {CKPT_PATH} -- run 05_Model_Training.ipynb first "
        f"to train and save the proposed model."
    )

model = model.to(device).eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")


Loaded checkpoint from epoch 195
  best val PSNR: 28.206
  best val SSIM: 0.7533
Model parameters: 116,138


## Core restoration function

In [4]:
@torch.no_grad()
def restore_image(noisy_np):
    """Runs the proposed model on one degraded image. Returns the restored
    image as a numpy array, clamped to the valid [0,1] intensity range."""
    x = torch.from_numpy(noisy_np.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(device)
    restored, mix_weights, beta, scale = model(x)
    restored_np = restored[0, 0].clamp(0, 1).cpu().numpy()
    return restored_np, mix_weights[0].cpu().numpy(), beta.cpu().numpy()


## Single-image demo

Shows Corrupt (input) -> Restored (output), plus a pixel-intensity
histogram comparison -- same visual language as the competition's own
"Hackathon Problem" slide, so this plot is directly usable for a
presentation/demo.

In [5]:
def show_before_after(noisy_np, restored_np, gt_np=None, title="", save_path=None):
    n_panels = 4 if gt_np is not None else 3
    fig, axes = plt.subplots(1, n_panels, figsize=(4.2*n_panels, 4.2))

    axes[0].imshow(noisy_np, cmap="gray"); axes[0].set_title("Corrupt (input)"); axes[0].axis("off")
    axes[1].imshow(restored_np, cmap="gray"); axes[1].set_title("Restored (our model)"); axes[1].axis("off")

    idx = 2
    if gt_np is not None:
        axes[2].imshow(gt_np, cmap="gray")
        p = psnr_np(restored_np, gt_np)
        s = ssim_np(restored_np, gt_np)
        axes[2].set_title(f"Ground Truth\nPSNR={p:.2f} SSIM={s:.3f}")
        axes[2].axis("off")
        idx = 3

    axes[idx].hist(noisy_np.flatten(), bins=100, alpha=0.5, density=True, label="Corrupt")
    axes[idx].hist(restored_np.flatten(), bins=100, alpha=0.5, density=True, label="Restored")
    if gt_np is not None:
        axes[idx].hist(gt_np.flatten(), bins=100, alpha=0.5, density=True, label="GT")
    axes[idx].set_title("Pixel intensity distribution")
    axes[idx].set_xlabel("pixel value"); axes[idx].legend(fontsize=8)

    fig.suptitle(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


test_paths = list_npy(TEST_NOISY_DIR)
print(f"Real competition test set: {len(test_paths)} images (no ground truth).")

sample_path = test_paths[0]
noisy_np = load_npy(sample_path)
restored_np, mix_weights, beta = restore_image(noisy_np)

show_before_after(noisy_np, restored_np,
                   title=f"Test sample: {os.path.basename(sample_path)}",
                   save_path="../results/demo_figures/single_demo.png")


Real competition test set: 400 images (no ground truth).


C:\Users\DELL\AppData\Local\Temp\ipykernel_34320\1900801871.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Multi-image demo grid (a handful of real test images)

In [6]:
N_DEMO = 4
fig, axes = plt.subplots(N_DEMO, 2, figsize=(8, 4*N_DEMO))
for row, p in enumerate(test_paths[:N_DEMO]):
    noisy_np = load_npy(p)
    restored_np, _, _ = restore_image(noisy_np)
    axes[row, 0].imshow(noisy_np, cmap="gray"); axes[row, 0].set_title(f"Corrupt: {os.path.basename(p)}"); axes[row, 0].axis("off")
    axes[row, 1].imshow(restored_np, cmap="gray"); axes[row, 1].set_title("Restored"); axes[row, 1].axis("off")
plt.tight_layout()
plt.savefig("../results/demo_figures/multi_demo_grid.png", dpi=150)
plt.show()


C:\Users\DELL\AppData\Local\Temp\ipykernel_34320\1225934821.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Quantitative sanity check (on labeled validation split)

The real competition test set has no ground truth, so we can't compute
PSNR/SSIM there. This section runs the same restoration on the held-out
VALIDATION split (which does have ground truth, from Notebook 05's split)
purely as a quantitative sanity check that the model is genuinely restoring
images well, not just producing plausible-looking output.

In [7]:
all_pairs = match_pairs(TRAIN_GT_DIR, TRAIN_NOISY_DIR)
_, val_pairs = split_pairs(all_pairs, val_frac=0.1, seed=42)   # same split used in Notebook 05
print(f"Validation set: {len(val_pairs)} labeled pairs.")

psnrs, ssims = [], []
for gt_path, noisy_path in val_pairs:
    gt_np = load_npy(gt_path)
    noisy_np = load_npy(noisy_path)
    restored_np, _, _ = restore_image(noisy_np)
    psnrs.append(psnr_np(restored_np, gt_np))
    ssims.append(ssim_np(restored_np, gt_np))

print(f"\nValidation-set sanity check:")
print(f"  Mean PSNR: {np.mean(psnrs):.3f} dB  (std {np.std(psnrs):.3f})")
print(f"  Mean SSIM: {np.mean(ssims):.4f}  (std {np.std(ssims):.4f})")


Validation set: 320 labeled pairs.

Validation-set sanity check:
  Mean PSNR: 28.206 dB  (std 4.987)
  Mean SSIM: 0.7533  (std 0.1742)


In [8]:
# One labeled example shown with full before/after/GT/histogram, for the demo
demo_gt_path, demo_noisy_path = val_pairs[0]
gt_np = load_npy(demo_gt_path)
noisy_np = load_npy(demo_noisy_path)
restored_np, _, _ = restore_image(noisy_np)

show_before_after(noisy_np, restored_np, gt_np=gt_np,
                   title=f"Labeled validation example: {os.path.basename(demo_noisy_path)}",
                   save_path="../results/demo_figures/labeled_demo.png")


C:\Users\DELL\AppData\Local\Temp\ipykernel_34320\1900801871.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Full batch inference on the REAL competition test set

This is the submission-ready deliverable: every one of the real test
images, restored and saved as `.npy` (same format as the input, same
base filename) into `results/test_predictions/`.

In [9]:
inference_times = []
n_processed = 0

t_start = time.time()
for p in test_paths:
    noisy_np = load_npy(p)

    t0 = time.time()
    restored_np, _, _ = restore_image(noisy_np)
    inference_times.append(time.time() - t0)

    out_name = os.path.basename(p)
    out_path = os.path.join(OUTPUT_DIR, out_name)
    np.save(out_path, restored_np.astype(np.float32))

    n_processed += 1
    if n_processed % 50 == 0:
        print(f"  processed {n_processed}/{len(test_paths)}")

total_time = time.time() - t_start


  processed 50/400
  processed 100/400
  processed 150/400
  processed 200/400
  processed 250/400
  processed 300/400
  processed 350/400
  processed 400/400


In [10]:
print("="*60)
print("BATCH INFERENCE COMPLETE")
print("="*60)
print(f"Images processed:        {n_processed}")
print(f"Total time:              {total_time:.1f} sec")
print(f"Mean per-image latency:  {np.mean(inference_times)*1000:.2f} ms")
print(f"Output directory:        {os.path.abspath(OUTPUT_DIR)}")
print(f"Checkpoint used:         {os.path.abspath(CKPT_PATH)}")


BATCH INFERENCE COMPLETE
Images processed:        400
Total time:              20.6 sec
Mean per-image latency:  22.41 ms
Output directory:        c:\Users\DELL\OneDrive\Desktop\Golu Kumar\Golu Kumar\Semi conductor image paper\results\test_predictions
Checkpoint used:         c:\Users\DELL\OneDrive\Desktop\Golu Kumar\Golu Kumar\Semi conductor image paper\results\checkpoints\DistributionMixtureRestorationNet.pth


In [12]:
import numpy as np
import matplotlib.pyplot as plt
import os

INPUT_DIR = r"C:\Users\DELL\OneDrive\Desktop\Golu Kumar\Golu Kumar\Semi conductor image paper\Test_NoisyLR\NoisyLR"  # your original noisy images
OUTPUT_DIR = r"C:\Users\DELL\OneDrive\Desktop\Golu Kumar\Golu Kumar\Semi conductor image paper\results\test_predictions"  # where inference.py saved the restored .npy files

# grab the first 10 restored files
restored_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith(".npy")])[:10]

fig, axes = plt.subplots(10, 2, figsize=(8, 40))
for i, fname in enumerate(restored_files):
    noisy = np.load(os.path.join(INPUT_DIR, fname))
    restored = np.load(os.path.join(OUTPUT_DIR, fname))

    axes[i, 0].imshow(noisy, cmap="gray")
    axes[i, 0].set_title(f"Noisy: {fname}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(restored, cmap="gray")
    axes[i, 1].set_title(f"Restored: {fname}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.savefig("../results/10_sample_restorations.png", dpi=100)
plt.show()
print("Saved: results/10_sample_restorations.png")

Saved: results/10_sample_restorations.png


C:\Users\DELL\AppData\Local\Temp\ipykernel_34320\369895265.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
import sys, os
sys.path.insert(0, "..")
from pathlib import Path

import numpy as np
import torch

from utils.data import list_npy, load_npy
from models.restoration_net import DistributionMixtureRestorationNet

device = "cuda" if torch.cuda.is_available() else "cpu"


PROJECT_ROOT = Path("..").resolve()

CKPT_PATH = (
    PROJECT_ROOT
    / "results"
    / "checkpoints"
    / "DistributionMixtureRestorationNet.pth"
)

model = DistributionMixtureRestorationNet(base_ch=32, n_components=3,
                                            n_lr_blocks=4, n_hr_blocks=2, use_film=True)
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model = model.to(device).eval()


TEST_NOISY_DIR = PROJECT_ROOT / "Test_NoisyLR" / "NoisyLR"

sample_paths = list_npy(TEST_NOISY_DIR)[:4]   
print(f"Checking {len(sample_paths)} images: {[os.path.basename(p) for p in sample_paths]}")

@torch.no_grad()
def restore_image(noisy_np):
    x = torch.from_numpy(noisy_np.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(device)
    restored, mix_weights, beta, scale = model(x)
    return restored[0, 0].clamp(0, 1).cpu().numpy(), mix_weights[0].cpu().numpy(), beta.cpu().numpy()


for img_path in sample_paths:
    noisy_np = load_npy(img_path)
    _, mix_weights, beta = restore_image(noisy_np)   # mix_weights: [K, H, W]

    print(f"\n{os.path.basename(img_path)}")
    for k in range(mix_weights.shape[0]):
        w = mix_weights[k]
        print(f"  component {k} (beta={beta[k]:.3f}): min={w.min():.4f}  max={w.max():.4f}  mean={w.mean():.4f}")

    dominant = mix_weights.argmax(0)
    counts = np.bincount(dominant.flatten(), minlength=mix_weights.shape[0])
    print(f"  pixel counts per dominant component: {counts}")

Checking 4 images: ['000000.npy', '000001.npy', '000002.npy', '000003.npy']

000000.npy
  component 0 (beta=1.734): min=0.0000  max=0.9981  mean=0.1045
  component 1 (beta=2.199): min=0.0000  max=0.9974  mean=0.4248
  component 2 (beta=2.499): min=0.0000  max=1.0000  mean=0.4707
  pixel counts per dominant component: [1445 7415 7524]

000001.npy
  component 0 (beta=1.734): min=0.0000  max=0.9999  mean=0.2218
  component 1 (beta=2.199): min=0.0000  max=0.9998  mean=0.5414
  component 2 (beta=2.499): min=0.0000  max=1.0000  mean=0.2368
  pixel counts per dominant component: [3401 9307 3676]

000002.npy
  component 0 (beta=1.734): min=0.0000  max=0.9986  mean=0.1397
  component 1 (beta=2.199): min=0.0000  max=0.9977  mean=0.4030
  component 2 (beta=2.499): min=0.0000  max=1.0000  mean=0.4573
  pixel counts per dominant component: [2287 6728 7369]

000003.npy
  component 0 (beta=1.734): min=0.0000  max=1.0000  mean=0.5575
  component 1 (beta=2.199): min=0.0000  max=1.0000  mean=0.3932
  co